# Can & Square: Hierarchical CEM Experiment

**Does hierarchical CEM work on longer-horizon tasks (Can, Square)?**

Replicates the Module B pipeline from `module_b_hierarchical_planning.ipynb` but for Can and Square Robomimic tasks.
Select task below, then run all cells sequentially.

| Step | Description |
|------|-------------|
| S1-S3 | Data download + DINOv2 pre-embedding |
| S4-S5 | Build K-triplets + train JEPA Transformer predictors (K=1,5,10,20) |
| S6-S7 | Load parquet + extract subgoals from demos |
| S8-S9 | Subgoal predictor training |
| S10-S13 | Flat CEM vs Hierarchical CEM evaluation |
| S14-S15 | Plots + summary |


In [1]:
# ── TASK SELECTOR ──
TASK = 'can'  # 'can' or 'square'
print(f'Selected task: {TASK.upper()}')

Selected task: CAN


In [2]:
import sys, subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch>=2.0', 'torchvision', 'matplotlib', 'scikit-learn', 'tqdm',
            'numpy', 'pandas', 'scipy', 'pyarrow>=15', 'pillow', 'av',
            'huggingface_hub', 'einops']:
    pip_install(pkg)

import os, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import av
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.signal import argrelextrema
from huggingface_hub import snapshot_download

# -- Paths (Colab-aware) --
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/jepa_action')
    print('Mounted Google Drive. Ensure data/ is at MyDrive/jepa_action/data/')
except ImportError:
    ROOT = Path(os.getcwd()).resolve()
    if not (ROOT / 'data' / 'embeddings').exists():
        if (ROOT.parent / 'data' / 'embeddings').exists():
            ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
EMBED_DIR = DATA_DIR / 'embeddings'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EMBED_DIR.mkdir(parents=True, exist_ok=True)

DATASET_DIR = DATA_DIR / f'robomimic-ph-{TASK}-image'
PARQUET_DIR = DATASET_DIR / 'data' / 'chunk-000'

# -- Config --
class Config:
    embed_dim = 384
    action_dim = 7
    d_model = 256
    n_heads = 4
    n_layers_tr = 4
    n_layers_sg = 2
    batch_size = 512      # Colab T4/A100: 15-24GB VRAM
    lr = 1e-4
    weight_decay = 0.05
    warmup_steps = 200
    max_epochs = 30
    use_amp = True
    dropout_p = 0.2
    sg_dropout = 0.1
    sg_hidden = [256, 128]
    image_size = 224
    subsample_rate = 2
    seed = 42
    K_values = [1, 5, 10, 20]
    K_values_hier = [1, 5]
    best_T = 8
    WG_values = [1]
    min_gap = 10
    cem_n_pop = 128
    cem_elite_frac = 0.1
    cem_n_iters = 5
    flat_H_values = [20, 40]
    hier_h_values = [5, 10]
    hier_sg_keys = ['Transf_WG1', 'MLP_WG1']
    success_threshold = 0.90
    n_train_episodes = 160
    n_cycles = 15 if TASK == 'can' else 20

cfg = Config()

# -- Device & reproducibility --
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Train episodes: {cfg.n_train_episodes}, Hier cycles: {cfg.n_cycles}')
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark = True

print('Setup complete.')


Mounted at /content/drive
Mounted Google Drive. Ensure data/ is at MyDrive/jepa_action/data/
Device: cuda
GPU: NVIDIA L4
Train episodes: 160, Hier cycles: 15
Setup complete.


## S2: Download Dataset from HuggingFace

Downloads the LeRobot v2.1 dataset for the selected task. Uses PyArrow (NOT pandas) for parquet.


In [ ]:
# -- Download LeRobot v2.1 dataset from HuggingFace --
# Repo: ankile/robomimic-ph-{TASK}-image (public, NOT gated).
# Same format as data/robomimic-ph-lift-image/ (200 episodes, Parquet + mp4, 84x84).
from huggingface_hub import snapshot_download

repo = f'ankile/robomimic-ph-{TASK}-image'

if DATASET_DIR.exists() and PARQUET_DIR.exists():
    print(f'Dataset already exists at {DATASET_DIR}')
else:
    print(f'Downloading {repo} from HuggingFace...')
    snapshot_download(
        repo_id=repo,
        repo_type='dataset',
        local_dir=str(DATASET_DIR),
        allow_patterns=['*.parquet', '*.json', '*.mp4'],
    )
    print('Download complete.')

# Show structure
for subdir in ['meta', 'data', 'videos']:
    p = DATASET_DIR / subdir
    if p.exists():
        count = len(list(p.rglob('*')))
        size_mb = sum(f.stat().st_size for f in p.rglob('*') if f.is_file()) / 1e6
        print(f'  {subdir}/: {count} files, {size_mb:.1f} MB')

## S3: DINOv2 ViT-S/14 Pre-Embedding

Load frames from MP4 videos via PyAV, run through frozen DINOv2 ViT-S/14, cache embeddings.


In [ ]:
# ── DINOv2 model ──
print('Loading DINOv2 ViT-S/14...')
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', trust_repo=True)
dinov2 = dinov2.to(device)
dinov2.eval()
if cfg.use_amp and device.type == 'cuda':
    dinov2 = dinov2.half()
for p in dinov2.parameters():
    p.requires_grad = False
print(f'DINOv2 ViT-S/14 loaded: {sum(p.numel() for p in dinov2.parameters())/1e6:.1f}M params')

# ── Image transform ──
dino_transform = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [ ]:
# ── Episode reader ──
def read_episode(episode_idx, camera='agentview'):
    chunk = episode_idx // 1000
    ep_str = f'episode_{episode_idx:06d}'
    # Parquet
    pq_path = PARQUET_DIR / f'{ep_str}.parquet'
    table = pq.read_table(str(pq_path))
    actions_list = table.column('action').to_pylist()
    actions = np.array(actions_list, dtype=np.float32)
    states_list = table.column('observation.state').to_pylist()
    states = np.array(states_list, dtype=np.float32)
    # Video
    video_key = f'observation.images.{camera}'
    video_path = DATASET_DIR / 'videos' / f'chunk-{chunk:03d}' / video_key / f'{ep_str}.mp4'
    frames = []
    container = av.open(str(video_path))
    for frame in container.decode(video=0):
        frames.append(frame.to_ndarray(format='rgb24'))
    container.close()
    frames = np.array(frames, dtype=np.uint8)
    m = min(len(frames), len(actions))
    return frames[:m], actions[:m], states[:m]

# Quick test
f0, a0, s0 = read_episode(0)
print(f'Episode 0: frames={f0.shape}, actions={a0.shape}, states={s0.shape}')


In [ ]:
# ── Extract all episodes (subsample 20Hz → 10Hz) ──
n_episodes = len(sorted(PARQUET_DIR.glob('episode_*.parquet')))
print(f'Extracting {n_episodes} episodes...')

all_frames, all_actions, all_states, ep_lengths = [], [], [], []
for ep in tqdm(range(n_episodes), desc='Reading episodes'):
    frames, actions, states = read_episode(ep)
    idx = np.arange(0, len(frames), cfg.subsample_rate)
    all_frames.append(frames[idx])
    all_actions.append(actions[idx])
    all_states.append(states[idx])
    ep_lengths.append(len(idx))

n_total = sum(ep_lengths)
print(f'Total frames (10Hz): {n_total}, avg/ep: {n_total/n_episodes:.1f}, min/max: {min(ep_lengths)}/{max(ep_lengths)}')


In [ ]:
# ── Train/val split (first N train, rest val) ──
train_eps = list(range(cfg.n_train_episodes))
val_eps = list(range(cfg.n_train_episodes, n_episodes))

train_frames_cat = np.concatenate([all_frames[i] for i in train_eps], axis=0)
val_frames_cat = np.concatenate([all_frames[i] for i in val_eps], axis=0)
train_acts_cat = np.concatenate([all_actions[i] for i in train_eps], axis=0)
val_acts_cat = np.concatenate([all_actions[i] for i in val_eps], axis=0)

print(f'Train: {len(train_frames_cat)} frames from {len(train_eps)} episodes')
print(f'Val:   {len(val_frames_cat)} frames from {len(val_eps)} episodes')

# ── Action normalization ──
action_mean = train_acts_cat.mean(axis=0, keepdims=False).astype(np.float32)
action_std = train_acts_cat.std(axis=0, keepdims=False).astype(np.float32)
action_std = np.where(action_std < 1e-8, 1.0, action_std)
norm_stats = {'mean': torch.tensor(action_mean), 'std': torch.tensor(action_std)}
torch.save(norm_stats, EMBED_DIR / f'norm_stats_{TASK}.pt')

train_acts_norm = (train_acts_cat - action_mean) / action_std
val_acts_norm = (val_acts_cat - action_mean) / action_std
print(f'Action norm stats saved.')


In [ ]:
# ── DINOv2 batch inference ──
@torch.no_grad()
def compute_embeddings(frames_array, batch_size=64, desc='Embedding'):
    n = len(frames_array)
    embeddings = np.zeros((n, cfg.embed_dim), dtype=np.float32)
    for i in tqdm(range(0, n, batch_size), desc=desc):
        batch_frames = frames_array[i:i+batch_size]
        tensors = []
        for frame in batch_frames:
            img = Image.fromarray(frame)
            tensors.append(dino_transform(img))
        batch = torch.stack(tensors).to(device)
        if cfg.use_amp and device.type == 'cuda':
            with autocast():
                emb = dinov2(batch)
        else:
            emb = dinov2(batch)
        embeddings[i:i+batch_size] = emb.cpu().numpy()
    return embeddings

# Compute and cache
EMB_TRAIN_PATH = EMBED_DIR / f'embeddings_{TASK}_train.pt'
EMB_VAL_PATH = EMBED_DIR / f'embeddings_{TASK}_val.pt'
ACT_TRAIN_PATH = EMBED_DIR / f'actions_{TASK}_train.pt'
ACT_VAL_PATH = EMBED_DIR / f'actions_{TASK}_val.pt'

if EMB_TRAIN_PATH.exists() and EMB_VAL_PATH.exists():
    print('Embeddings already cached.')
else:
    print('Computing train embeddings...')
    train_embs = compute_embeddings(train_frames_cat, desc='Train emb')
    print('Computing val embeddings...')
    val_embs = compute_embeddings(val_frames_cat, desc='Val emb')
    torch.save(torch.tensor(train_embs), EMB_TRAIN_PATH)
    torch.save(torch.tensor(val_embs), EMB_VAL_PATH)
    torch.save(torch.tensor(train_acts_norm), ACT_TRAIN_PATH)
    torch.save(torch.tensor(val_acts_norm), ACT_VAL_PATH)
    print(f'Embeddings saved: train={train_embs.shape}, val={val_embs.shape}')

# Free DINOv2 memory
del dinov2
torch.cuda.empty_cache()
print('DINOv2 released.')


## S4: Build K-Triplets

Build per-demographic-offset triplets for K ∈ {1, 5, 10, 20}.


In [ ]:
# Load cached embeddings
train_emb = torch.load(EMB_TRAIN_PATH).float()
val_emb = torch.load(EMB_VAL_PATH).float()
train_act = torch.load(ACT_TRAIN_PATH).float()
val_act = torch.load(ACT_VAL_PATH).float()
print(f'Train: {train_emb.shape}, Val: {val_emb.shape}')

mean_embed = train_emb.mean(dim=0, keepdim=True)


In [ ]:
# ── Build offsets ──
def build_offsets(frame_counts):
    offsets = []
    cum = 0
    for n in frame_counts:
        offsets.append((cum, n))
        cum += n
    return offsets

train_lengths = [len(all_frames[i]) for i in train_eps]
val_lengths = [len(all_frames[i]) for i in val_eps]
train_offsets = build_offsets(train_lengths)
val_offsets = build_offsets(val_lengths)

def build_triplets(offsets, K):
    triplets = []
    for start, length in offsets:
        for t in range(length - K):
            triplets.append([start + t, start + t, start + t + K])
    return torch.tensor(triplets, dtype=torch.long)

for K in cfg.K_values:
    tp = EMBED_DIR / f'triplets_{TASK}_train_K{K}.pt'
    tv = EMBED_DIR / f'triplets_{TASK}_val_K{K}.pt'
    if not tp.exists():
        torch.save(build_triplets(train_offsets, K), tp)
        torch.save(build_triplets(val_offsets, K), tv)
    print(f'K={K}: train={len(torch.load(tp)):,}, val={len(torch.load(tv)):,}')


## S5: Train JEPA Transformer Predictors

Re-use TransformerPredictor and training loop from `module_a_predictor.ipynb`.
Train for K ∈ {1, 5, 10, 20}. Cached in `outputs/`.


In [ ]:
# ── Model definitions (from module_a_predictor.ipynb) ──
class ActionMLP(nn.Module):
    def __init__(self, action_dim=7, hidden_dim=128, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(action_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, a): return self.net(a)

class TransformerPredictor(nn.Module):
    def __init__(self, embed_dim=384, action_dim=7, d_model=256,
                 n_layers=4, n_heads=4, max_seq_len=64, dropout=0.2):
        super().__init__()
        self.d_model = d_model
        self.embed_proj = nn.Linear(embed_dim, d_model)
        self.action_mlp = ActionMLP(action_dim, hidden_dim=128, out_dim=d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, batch_first=True,
            dropout=dropout, activation='gelu')
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.predictor = nn.Linear(d_model, embed_dim)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)

    def forward(self, embeds, actions):
        if embeds.dim() == 2:
            embeds = embeds.unsqueeze(1); actions = actions.unsqueeze(1)
        B, T, _ = embeds.shape
        x = self.embed_proj(embeds) + self.action_mlp(actions)
        x = x + self.pos_embed[:, :T, :]
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        x = self.transformer(x, mask=causal_mask)
        return self.predictor(x[:, -1, :])

print(f'Transformer params: {sum(p.numel() for p in TransformerPredictor().parameters()):,}')


In [ ]:
# ── Dataset + training (from module_a_predictor.ipynb) ──
class ContextTripletDataset(Dataset):
    def __init__(self, emb, act, trip, T=1):
        self.embeddings = emb
        self.actions = act
        self.triplets = trip
        self.T = T
        valid = self.triplets[:, 0] >= (T - 1)
        self.triplets = self.triplets[valid]

    def __len__(self): return len(self.triplets)

    def __getitem__(self, idx):
        t_idx, a_idx, tpK_idx = self.triplets[idx]
        z_seq = self.embeddings[t_idx - self.T + 1 : t_idx + 1]
        a_seq = self.actions[t_idx - self.T + 1 : t_idx + 1]
        target = self.embeddings[tpK_idx]
        if z_seq.shape[0] < self.T:
            pad = self.T - z_seq.shape[0]
            z_seq = torch.cat([torch.zeros(pad, self.embeddings.shape[1]), z_seq], dim=0)
            a_seq = torch.cat([torch.zeros(pad, self.actions.shape[1]), a_seq], dim=0)
        return z_seq, a_seq, target

def cosine_loss(pred, target):
    return 1.0 - F.cosine_similarity(pred, target, dim=-1).mean()

def get_lr(step, warmup_steps, total_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

@torch.no_grad()
def compute_val_cosine(model, loader):
    model.eval()
    cs_list = []
    for z_seq, a_seq, target in loader:
        z_seq, a_seq, target = z_seq.to(device), a_seq.to(device), target.to(device)
        if cfg.use_amp and device.type == 'cuda':
            with autocast(): pred = model(z_seq, a_seq)
        else:
            pred = model(z_seq, a_seq)
        cs_list.append(F.cosine_similarity(pred, target, dim=-1).mean().item())
    return np.mean(cs_list)

def train_model(model, K, T, epochs=None, verbose=False):
    epochs = epochs or cfg.max_epochs
    train_emb = torch.load(EMB_TRAIN_PATH)
    train_act = torch.load(ACT_TRAIN_PATH)
    val_emb = torch.load(EMB_VAL_PATH)
    val_act = torch.load(ACT_VAL_PATH)
    train_trip = torch.load(EMBED_DIR / f'triplets_{TASK}_train_K{K}.pt')
    val_trip = torch.load(EMBED_DIR / f'triplets_{TASK}_val_K{K}.pt')

    train_ds = ContextTripletDataset(train_emb, train_act, train_trip, T)
    val_ds = ContextTripletDataset(val_emb, val_act, val_trip, T)
    train_ldr = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, pin_memory=(device.type=='cuda'), drop_last=True)
    val_ldr = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, pin_memory=(device.type=='cuda'))

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = GradScaler(enabled=(cfg.use_amp and device.type=='cuda'))
    total_steps = epochs * len(train_ldr)
    gs = 0
    best_cos = -1.0
    val_cos_list = []

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for z_seq, a_seq, target in train_ldr:
            z_seq, a_seq, target = z_seq.to(device), a_seq.to(device), target.to(device)
            lr = get_lr(gs, cfg.warmup_steps, total_steps, cfg.lr)
            for pg in optimizer.param_groups: pg['lr'] = lr
            if cfg.use_amp and device.type == 'cuda':
                with autocast():
                    pred = model(z_seq, a_seq)
                    loss = cosine_loss(pred, target)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            else:
                pred = model(z_seq, a_seq)
                loss = cosine_loss(pred, target)
                loss.backward(); optimizer.step()
            optimizer.zero_grad()
            epoch_loss += loss.item()
            gs += 1
        val_cos = compute_val_cosine(model, val_ldr)
        val_cos_list.append(val_cos)
        if val_cos > best_cos: best_cos = val_cos
        if verbose or epoch % max(1, epochs//5) == 0 or epoch == 1 or epoch == epochs:
            print(f'E {epoch:3d}/{epochs} | loss={epoch_loss/len(train_ldr):.4f} | val_cos={val_cos:.4f} | best={best_cos:.4f}')
    return {'best_cos': best_cos, 'val_cos_list': val_cos_list}

print('Training utilities ready.')


In [ ]:
# ── Train all K predictors ──
for K in cfg.K_values:
    ckpt_path = OUTPUT_DIR / f'transformer_{TASK}_K{K}_T{cfg.best_T}.pt'
    if ckpt_path.exists():
        print(f'[CACHED] K={K}')
        continue
    print(f'\n{"="*50}')
    print(f'Training Transformer K={K}')
    print(f'{"="*50}')
    model = TransformerPredictor(
        embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
        n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
    ).to(device)
    result = train_model(model, K, cfg.best_T, verbose=True)
    torch.save({
        'model_state_dict': model.state_dict(),
        'best_cos': result['best_cos'],
        'K': K, 'T': cfg.best_T,
    }, ckpt_path)
    print(f'Saved: {ckpt_path}')

print('\nAll predictors trained.')


## S6: Load Parquet + Extract Per-Episode Gripper/Action Data

In [ ]:
# Load all parquet files, extract gripper + actions at 10Hz
parquet_files = sorted(PARQUET_DIR.glob('episode_*.parquet'))
print(f'Found {len(parquet_files)} parquet episodes')

all_ep_actions = []
all_ep_gripper = []
all_ep_n_frames = []

for pf in tqdm(parquet_files, desc='Loading parquets'):
    df = pd.read_parquet(pf)
    actions_raw = np.stack(df['action'].values).astype(np.float32)
    states_raw = np.stack(df['observation.state'].values).astype(np.float32)
    gripper_openness = states_raw[:, 7] - states_raw[:, 8]
    all_ep_actions.append(torch.tensor(actions_raw[::2]))
    all_ep_gripper.append(torch.tensor(gripper_openness[::2]))
    all_ep_n_frames.append(len(actions_raw[::2]))

# Verify alignment
train_frames_total = sum(all_ep_n_frames[:cfg.n_train_episodes])
val_frames_total = sum(all_ep_n_frames[cfg.n_train_episodes:])
print(f'Train frames (10Hz): {train_frames_total} (emb: {train_emb.shape[0]})')
print(f'Val frames (10Hz):   {val_frames_total} (emb: {val_emb.shape[0]})')
assert train_frames_total == train_emb.shape[0], 'Train mismatch!'
assert val_frames_total == val_emb.shape[0], 'Val mismatch!'

# Episode → embedding index mapping
train_offset = 0; val_offset = 0
ep_to_emb_start = {}
for ep_idx in range(len(parquet_files)):
    n_frames = all_ep_n_frames[ep_idx]
    if ep_idx < cfg.n_train_episodes:
        ep_to_emb_start[ep_idx] = ('train', train_offset)
        train_offset += n_frames
    else:
        ep_to_emb_start[ep_idx] = ('val', val_offset)
        val_offset += n_frames

def get_emb_idx(ep_idx, local_frame_idx):
    split, start = ep_to_emb_start[ep_idx]
    return split, start + local_frame_idx

print('Episode mapping built.')


## S7: B1 - Subgoal Extraction

In [ ]:
def detect_gripper_transitions(gripper_openness, percentile=95):
    if len(gripper_openness) < 2: return []
    g = gripper_openness.numpy() if isinstance(gripper_openness, torch.Tensor) else gripper_openness
    diffs = np.abs(np.diff(g))
    threshold = np.percentile(diffs, percentile)
    if threshold <= 0: return []
    change_frames = np.where(diffs > threshold)[0] + 1
    return sorted(change_frames.tolist())

def detect_velocity_peaks(actions, n_peaks=3):
    a = actions.numpy() if isinstance(actions, torch.Tensor) else actions
    vel_mag = np.linalg.norm(a, axis=1)
    if len(vel_mag) < 3: return []
    peaks = argrelextrema(vel_mag, np.greater, order=2)[0]
    if len(peaks) == 0: return []
    if n_peaks is not None and len(peaks) > n_peaks:
        peak_vals = vel_mag[peaks]
        top_idx = np.argsort(peak_vals)[-n_peaks:]
        peaks = peaks[top_idx]
    return sorted(peaks.tolist())

def extract_subgoals(actions, gripper_openness, min_gap=10, percentile=95, n_peaks=3):
    gripper_frames = detect_gripper_transitions(gripper_openness, percentile)
    vel_frames = detect_velocity_peaks(actions, n_peaks)
    candidates = sorted(set(gripper_frames + vel_frames))
    subgoals = []
    for idx in candidates:
        if not subgoals or (idx - subgoals[-1]) >= min_gap:
            subgoals.append(idx)
    return subgoals

print('Subgoal extraction functions defined.')


In [ ]:
SUBGOALS_CACHE = EMBED_DIR / f'subgoals_{TASK}.pt'

if SUBGOALS_CACHE.exists():
    print('Loading cached subgoals...')
    d = torch.load(SUBGOALS_CACHE, weights_only=False)
    train_sg_local = d['train_sg_local']
    train_sg_embs = d['train_sg_embs']
else:
    print('Extracting subgoals...')
    train_sg_local = []
    train_sg_embs = []
    for ep in tqdm(range(cfg.n_train_episodes), desc='Train subgoals'):
        actions = all_ep_actions[ep]
        gripper = all_ep_gripper[ep]
        n_frames = all_ep_n_frames[ep]
        sg_local = extract_subgoals(actions, gripper, cfg.min_gap)
        if len(sg_local) == 0:
            sg_local = [n_frames // 2]
        train_sg_local.append(sg_local)
    for ep in range(cfg.n_train_episodes):
        sg_local_ep = train_sg_local[ep]
        _, start = ep_to_emb_start[ep]
        sg_embs_ep = train_emb[torch.tensor([start + idx for idx in sg_local_ep])]
        train_sg_embs.append(sg_embs_ep)
    torch.save({'train_sg_local': train_sg_local, 'train_sg_embs': train_sg_embs}, SUBGOALS_CACHE)

sg_counts = [len(sg) for sg in train_sg_local]
print(f'\nB1: Episodes with subgoals: {sum(1 for c in sg_counts if c > 0)}/{cfg.n_train_episodes}')
print(f'     Total subgoals: {sum(sg_counts)}, Mean/ep: {np.mean(sg_counts):.2f}')


## S8: B2 - Subgoal Visualization

In [ ]:
n_viz = min(5, cfg.n_train_episodes)
viz_eps = [0, 15, 40, 80, 120][:n_viz] if cfg.n_train_episodes > 80 else list(range(min(5, cfg.n_train_episodes)))

fig, axes = plt.subplots(n_viz, 2, figsize=(14, 3 * n_viz))
if n_viz == 1: axes = axes.reshape(1, -1)

for i, ep in enumerate(viz_eps):
    actions = all_ep_actions[ep].numpy()
    gripper = all_ep_gripper[ep].numpy()
    sg = train_sg_local[ep]
    vel_mag = np.linalg.norm(actions, axis=1)
    frames = np.arange(len(actions))

    for j, (label, color, data) in enumerate([('Gripper', 'b', gripper), ('||Action||', 'g', vel_mag)]):
        ax = axes[i, j]
        ax.plot(frames, data, c=color, linewidth=1.5, alpha=0.7)
        for s in sg: ax.axvline(x=s, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
        ax.set_ylabel(label)
        if i == n_viz - 1: ax.set_xlabel('Frame (10Hz)')
        ax.set_title(f'E{ep} - {label}' + (f' ({len(sg)} sg)' if j == 1 else ''))
        ax.grid(True, alpha=0.3)

fig.suptitle(f'B2: Subgoal Extraction ({TASK.upper()})', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'b2_subgoal_{TASK}.png', dpi=150, bbox_inches='tight')
plt.show()


## S9: B3 - Subgoal Predictor

In [ ]:
class SubgoalDataset(Dataset):
    def __init__(self, subgoal_embeddings_list, WG=1):
        self.samples = []
        for sg_embs in subgoal_embeddings_list:
            M = sg_embs.shape[0]
            for i in range(WG, M):
                context = sg_embs[i - WG:i].flatten()
                target = sg_embs[i]
                self.samples.append((context, target))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

class MLPSubgoalPredictor(nn.Module):
    def __init__(self, input_dim, embed_dim=384, hidden=[256, 128]):
        super().__init__()
        layers = []; in_dim = input_dim
        for h in hidden: layers.extend([nn.Linear(in_dim, h), nn.ReLU()]); in_dim = h
        layers.append(nn.Linear(in_dim, embed_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class TransformerSubgoalPredictor(nn.Module):
    def __init__(self, embed_dim=384, d_model=256, n_heads=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.d_model = d_model; self.embed_dim = embed_dim
        self.embed_proj = nn.Linear(embed_dim, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, 10, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, batch_first=True, dropout=dropout, activation='gelu')
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.predictor = nn.Linear(d_model, embed_dim)
    def forward(self, x):
        B, D = x.shape; WG = D // self.embed_dim
        x = x.view(B, WG, self.embed_dim)
        x = self.embed_proj(x) + self.pos_embed[:, :WG, :]
        x = self.transformer(x)
        return self.predictor(x[:, -1, :])

print(f'MLP params: {sum(p.numel() for p in MLPSubgoalPredictor(384).parameters()):,}')
print(f'Transformer params: {sum(p.numel() for p in TransformerSubgoalPredictor().parameters()):,}')


In [ ]:
n_sg_train = int(len(train_sg_embs) * 0.8)
sg_embs_tr = train_sg_embs[:n_sg_train]
sg_embs_val = train_sg_embs[n_sg_train:]
print(f'SG train episodes: {n_sg_train}, val episodes: {len(train_sg_embs) - n_sg_train}')
print(f'Subgoals: train={sum(len(s) for s in sg_embs_tr)}, val={sum(len(s) for s in sg_embs_val)}')

def sg_train_model(model, train_loader, val_loader, epochs=30, verbose=False):
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = GradScaler(enabled=(cfg.use_amp and device.type == 'cuda'))
    total_steps = epochs * len(train_loader)
    gs = 0; best_cos = -1.0; val_cos_list = []
    for epoch in range(1, epochs + 1):
        model.train(); epoch_loss = 0.0
        for ctx, target in train_loader:
            ctx, target = ctx.to(device), target.to(device)
            lr = get_lr(gs, cfg.warmup_steps, total_steps, cfg.lr)
            for pg in optimizer.param_groups: pg['lr'] = lr
            if cfg.use_amp and device.type == 'cuda':
                with autocast():
                    pred = model(ctx); loss = cosine_loss(pred, target)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                pred = model(ctx); loss = cosine_loss(pred, target)
                loss.backward(); optimizer.step()
            optimizer.zero_grad(); epoch_loss += loss.item(); gs += 1
        model.eval(); cs_list = []
        for ctx, target in val_loader:
            ctx, target = ctx.to(device), target.to(device)
            if cfg.use_amp and device.type == 'cuda':
                with autocast(): pred = model(ctx)
            else: pred = model(ctx)
            cs_list.append(F.cosine_similarity(pred, target, dim=-1).mean().item())
        val_cos = np.mean(cs_list); val_cos_list.append(val_cos)
        if val_cos > best_cos: best_cos = val_cos
        if verbose or epoch % max(1, epochs//5) == 0 or epoch == 1 or epoch == epochs:
            print(f'E {epoch:3d}/{epochs} | loss={epoch_loss/len(train_loader):.4f} | val_cos={val_cos:.4f} | best={best_cos:.4f}')
    return {'val_cos': val_cos, 'best_cos': best_cos, 'val_cos_list': val_cos_list}

sg_results = {}
for arch_name, make_fn in [
    ('MLP', lambda d: MLPSubgoalPredictor(d, cfg.embed_dim, cfg.sg_hidden)),
    ('Transf', lambda d: TransformerSubgoalPredictor(cfg.embed_dim, cfg.d_model, cfg.n_heads, cfg.n_layers_sg, cfg.sg_dropout)),
]:
    for WG in cfg.WG_values:
        key = f'{arch_name}_WG{WG}'
        ckpt_path = OUTPUT_DIR / f'b3_{TASK}_{key}.pt'
        if ckpt_path.exists():
            print(f'[CACHED] {key}')
            d = torch.load(ckpt_path, map_location=device, weights_only=False)
            sg_results[key] = d
            continue
        input_dim = WG * cfg.embed_dim
        train_ds = SubgoalDataset(sg_embs_tr, WG)
        val_ds = SubgoalDataset(sg_embs_val, WG)
        if len(train_ds) == 0:
            print(f'  No samples for {key}, skipping.')
            continue
        train_ldr = DataLoader(train_ds, batch_size=min(64, len(train_ds)), shuffle=True)
        val_ldr = DataLoader(val_ds, batch_size=64, shuffle=False)
        model = make_fn(input_dim).to(device)
        result = sg_train_model(model, train_ldr, val_ldr, cfg.max_epochs, verbose=True)
        torch.save({
            'model_state_dict': model.state_dict(),
            'val_cos': result['val_cos'], 'best_cos': result['best_cos'],
            'WG': WG, 'arch': arch_name,
        }, ckpt_path)
        sg_results[key] = {'val_cos': result['val_cos'], 'best_cos': result['best_cos'], 'WG': WG, 'arch': arch_name}

best_sg_key = max(sg_results, key=lambda k: sg_results[k]['best_cos'])
print(f'\nBest subgoal predictor: {best_sg_key} (best_cos={sg_results[best_sg_key]["best_cos"]:.4f})')


## S10: Load JEPA Predictors

In [ ]:
jepa_predictors = {}
for K in cfg.K_values:
    ckpt_path = OUTPUT_DIR / f'transformer_{TASK}_K{K}_T{cfg.best_T}.pt'
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = TransformerPredictor(
        embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
        n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    jepa_predictors[K] = model
    print(f'Loaded K={K}')


## S11: CEM Planner Functions

In [ ]:
@torch.no_grad()
def cem_plan(z_start, z_target, predictor, horizon, K, T,
             n_pop=128, elite_frac=0.1, n_iters=5, noise_scale=0.5,
             action_dim=7, verbose=False, z_secondary=None, alpha=1.0):
    n_elite = max(1, int(n_pop * elite_frac))
    mu = torch.zeros(horizon, action_dim, device=device)
    sigma = torch.ones(horizon, action_dim, device=device) * noise_scale
    best_seq, best_score = None, -float('inf')

    for it in range(n_iters):
        eps = torch.randn(n_pop, horizon, action_dim, device=device)
        actions = mu.unsqueeze(0) + sigma.unsqueeze(0) * eps
        scores = torch.zeros(n_pop, device=device)

        for i in range(n_pop):
            z_ctx = z_start.clone().unsqueeze(0).unsqueeze(0).repeat(1, T, 1)
            a_ctx = torch.zeros(1, T, action_dim, device=device)
            a_ctx[:, -1, :] = actions[i, 0]
            for step in range(horizon):
                a_cur = actions[i, step:step+1].unsqueeze(0)
                a_ctx = torch.cat([a_ctx[:, 1:, :], a_cur], dim=1)
                pred = predictor(z_ctx, a_ctx)
                z_ctx = torch.cat([z_ctx[:, 1:, :], pred.unsqueeze(1)], dim=1)
            z_pred = z_ctx[:, -1, :].squeeze(0)
            cos_goal = F.cosine_similarity(z_pred.unsqueeze(0), z_target.unsqueeze(0), dim=-1)
            if z_secondary is not None:
                cos_secondary = F.cosine_similarity(z_pred.unsqueeze(0), z_secondary.unsqueeze(0), dim=-1)
                scores[i] = alpha * cos_goal + (1 - alpha) * cos_secondary
            else:
                scores[i] = cos_goal

        elite_idx = torch.topk(scores, n_elite).indices
        ea = actions[elite_idx]
        mu = ea.mean(dim=0)
        sigma = ea.std(dim=0) + 1e-6
        if scores[elite_idx[0]].item() > best_score:
            best_score = scores[elite_idx[0]].item()
            best_seq = ea[0].cpu().numpy()
    return best_seq, best_score

@torch.no_grad()
def flat_cem_plan(z_start, z_goal, predictor, H, K, T, **kw):
    horizon = max(1, H // K)
    return cem_plan(z_start, z_goal, predictor, horizon, K, T, **kw)

@torch.no_grad()
def hierarchical_cem_plan(z_start, sg_model, jepa_pred, h, K, T,
                          n_cycles=10, z_goal=None, alpha_start=0.5, alpha_end=0.95,
                          early_stop_thresh=0.95, **kw):
    WG = getattr(sg_model, 'WG', 1)
    z_cur = z_start.clone()
    past_sgs = [z_cur.cpu().numpy()]

    for cycle in range(n_cycles):
        while len(past_sgs) < WG:
            past_sgs.append(past_sgs[-1])
        ctx = torch.tensor(np.concatenate(past_sgs[-WG:])).float().unsqueeze(0).to(device)
        z_sg = sg_model(ctx).squeeze(0)

        progress = cycle / max(1, n_cycles - 1)
        alpha = alpha_start + (alpha_end - alpha_start) * progress
        z_secondary = z_goal if z_goal is not None else None
        actions, score = cem_plan(z_cur, z_sg, jepa_pred, h, K, T,
                                  z_secondary=z_secondary, alpha=alpha, **kw)

        z_ctx = z_cur.unsqueeze(0).unsqueeze(0).repeat(1, T, 1)
        a_ctx = torch.zeros(1, T, cfg.action_dim, device=device)
        a_ctx[:, -1, :] = torch.tensor(actions[0]).float().to(device)
        for step in range(h):
            a_cur = torch.tensor(actions[step:step+1]).float().unsqueeze(0).to(device)
            a_ctx = torch.cat([a_ctx[:, 1:, :], a_cur], dim=1)
            pred = jepa_pred(z_ctx, a_ctx)
            z_ctx = torch.cat([z_ctx[:, 1:, :], pred.unsqueeze(1)], dim=1)
            z_cur = pred.squeeze(0)

        past_sgs.append(z_sg.cpu().numpy())
        if z_goal is not None:
            cur_cos = F.cosine_similarity(z_cur.unsqueeze(0), z_goal.unsqueeze(0), dim=-1).item()
            if cur_cos >= early_stop_thresh:
                break
    return z_cur

print('CEM planners defined.')


## S12: B5 - Flat CEM Evaluation

In [ ]:
eval_ep_indices = list(range(cfg.n_train_episodes, len(parquet_files)))
eval_samples = []
for ep in eval_ep_indices:
    n_frames = all_ep_n_frames[ep]
    if n_frames < 3: continue
    _, start = ep_to_emb_start[ep]
    z_s = val_emb[start]
    z_g = val_emb[start + n_frames - 1]
    eval_samples.append({'ep': ep, 'z_start': z_s, 'z_goal': z_g, 'n_frames': n_frames})
print(f'Eval samples: {len(eval_samples)} val episodes')


In [ ]:
FLAT_CACHE = EMBED_DIR / f'flat_cem_eval_{TASK}.pt'

if FLAT_CACHE.exists():
    print('Loading cached flat CEM results...')
    flat_cem_results = torch.load(FLAT_CACHE, weights_only=False)
else:
    flat_cem_results = {}
    for K in cfg.K_values:
        predictor = jepa_predictors[K]
        for H in cfg.flat_H_values:
            horizon = max(1, H // K)
            key = f'flat_K{K}_H{H}'
            print(f'\nFlat CEM: K={K}, H={H} (horizon={horizon})')
            goal_cos_list, success_list = [], []
            for sample in tqdm(eval_samples, desc=f'Flat K={K} H={H}'):
                z_s, z_g = sample['z_start'].to(device), sample['z_goal'].to(device)
                _, score = cem_plan(z_s, z_g, predictor, horizon, K, cfg.best_T,
                                   cfg.cem_n_pop, cfg.cem_elite_frac, cfg.cem_n_iters)
                goal_cos_list.append(score)
                success_list.append(1.0 if score >= cfg.success_threshold else 0.0)
            flat_cem_results[key] = {'goal_cos': np.mean(goal_cos_list), 'success_rate': np.mean(success_list)}
            print(f'  Goal cos={flat_cem_results[key]["goal_cos"]:.4f}, Success@{cfg.success_threshold:.2f}={flat_cem_results[key]["success_rate"]:.3f}')
    torch.save(flat_cem_results, FLAT_CACHE)


## S13: B5 - Hierarchical CEM Evaluation

In [ ]:
HIER_CACHE = EMBED_DIR / f'hier_cem_eval_{TASK}.pt'

if HIER_CACHE.exists():
    print('Loading cached hierarchical CEM results...')
    hier_cem_results = torch.load(HIER_CACHE, weights_only=False)
else:
    hier_cem_results = {}
    for K in cfg.K_values_hier:
        predictor = jepa_predictors[K]
        for sg_key in cfg.hier_sg_keys:
            r = sg_results[sg_key]
            for h in cfg.hier_h_values:
                key = f'hier_K{K}_h{h}_{sg_key}_weighted'
                if key in hier_cem_results: continue

                ckpt = torch.load(OUTPUT_DIR / f'b3_{TASK}_{sg_key}.pt', map_location=device, weights_only=False)
                if r['arch'] == 'MLP':
                    input_dim = r['WG'] * cfg.embed_dim
                    sg_model = MLPSubgoalPredictor(input_dim, cfg.embed_dim, cfg.sg_hidden).to(device)
                else:
                    sg_model = TransformerSubgoalPredictor(cfg.embed_dim, cfg.d_model, cfg.n_heads, cfg.n_layers_sg, cfg.sg_dropout).to(device)
                sg_model.load_state_dict(ckpt['model_state_dict'])
                sg_model.eval()
                sg_model.WG = r['WG']

                goal_cos_list, success_list = [], []
                for sample in tqdm(eval_samples, desc=key):
                    z_s, z_g = sample['z_start'].to(device), sample['z_goal'].to(device)
                    z_final = hierarchical_cem_plan(
                        z_s, sg_model, predictor, h, K, cfg.best_T,
                        n_cycles=cfg.n_cycles, z_goal=z_g,
                        alpha_start=0.5, alpha_end=0.95, early_stop_thresh=0.95,
                        n_pop=cfg.cem_n_pop, elite_frac=cfg.cem_elite_frac, n_iters=cfg.cem_n_iters)
                    score = F.cosine_similarity(z_final.unsqueeze(0), z_g.unsqueeze(0), dim=-1).item()
                    goal_cos_list.append(score)
                    success_list.append(1.0 if score >= cfg.success_threshold else 0.0)

                hier_cem_results[key] = {'goal_cos': np.mean(goal_cos_list), 'success_rate': np.mean(success_list)}
                print(f'  {key}: goal_cos={hier_cem_results[key]["goal_cos"]:.4f}, success={hier_cem_results[key]["success_rate"]:.3f}')
    torch.save(hier_cem_results, HIER_CACHE)

print('\nHierarchical CEM evaluation complete.')


## S14: B5 - Flat vs Hierarchical CEM Plots & Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
for Hi, H in enumerate(cfg.flat_H_values):
    flat_cos = [flat_cem_results.get(f'flat_K{k}_H{H}', {}).get('goal_cos', 0) for k in cfg.K_values]
    ax.plot(cfg.K_values, flat_cos, 'o-', linewidth=2, markersize=8, label=f'Flat H={H}', color=['#e41a1c', '#d62728'][Hi])
for h in cfg.hier_h_values:
    key_pfx = f'hier_K{{}}_h{h}_{best_sg_key}_weighted'
    hier_cos = [hier_cem_results.get(key_pfx.format(k), {}).get('goal_cos', 0) for k in cfg.K_values_hier]
    ax.plot(cfg.K_values_hier, hier_cos, 's--', linewidth=2, markersize=8, label=f'Hier h={h} ({best_sg_key})',
            color=['#377eb8', '#4daf4a'][cfg.hier_h_values.index(h)])
ax.set_xlabel('Prediction Horizon K'); ax.set_ylabel('Goal Cosine Similarity')
ax.set_title(f'B5: Flat vs Hierarchical CEM - {TASK.upper()}')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
for Hi, H in enumerate(cfg.flat_H_values):
    flat_sr = [flat_cem_results.get(f'flat_K{k}_H{H}', {}).get('success_rate', 0) for k in cfg.K_values]
    ax.plot(cfg.K_values, flat_sr, 'o-', linewidth=2, markersize=8, label=f'Flat H={H}', color=['#e41a1c', '#d62728'][Hi])
for h in cfg.hier_h_values:
    key_pfx = f'hier_K{{}}_h{h}_{best_sg_key}_weighted'
    hier_sr = [hier_cem_results.get(key_pfx.format(k), {}).get('success_rate', 0) for k in cfg.K_values_hier]
    ax.plot(cfg.K_values_hier, hier_sr, 's--', linewidth=2, markersize=8, label=f'Hier h={h} ({best_sg_key})',
            color=['#377eb8', '#4daf4a'][cfg.hier_h_values.index(h)])
ax.axhline(y=1.0, color='green', linestyle=':', alpha=0.5)
ax.set_xlabel('Prediction Horizon K'); ax.set_ylabel(f'Success@{(int(cfg.success_threshold*100))}')
ax.set_title(f'B5: Success Rate - {TASK.upper()}')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'b5_flat_vs_hier_{TASK}.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
print(f'\n{"="*80}')
print(f'B5: Flat vs Hierarchical CEM - {TASK.upper()} Summary')
print(f'{"="*80}')
print(f'\n{"Method":<35}{"K":<6}{"H/h":<8}{"Goal Cos":<12}{"Success@{:d}":<12}'.format(int(cfg.success_threshold*100)))
print('-' * 73)
for K in cfg.K_values:
    for H in cfg.flat_H_values:
        r = flat_cem_results.get(f'flat_K{K}_H{H}', {})
        print(f'{"Flat CEM":<35}{K:<6}{H:<8}{r.get("goal_cos", 0):<12.4f}{r.get("success_rate", 0):<12.3f}')
for h in cfg.hier_h_values:
    for K in cfg.K_values_hier:
        key = f'hier_K{K}_h{h}_{best_sg_key}_weighted'
        r = hier_cem_results.get(key, {})
        print(f'{"Hier CEM (best sg):":<35}{K:<6}{h:<8}{r.get("goal_cos", 0):<12.4f}{r.get("success_rate", 0):<12.3f}')


## S15: B6 - Subgoal Predictor Ablation on CEM (K=1)

In [ ]:
K_eval = 1
sorted_sg_keys = sorted(sg_results.keys())
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(sorted_sg_keys))
width = 0.35

for hi, h in enumerate(cfg.hier_h_values):
    cos_vals_h = [hier_cem_results.get(f'hier_K{K_eval}_h{h}_{k}_weighted', {}).get('goal_cos', 0) for k in sorted_sg_keys]
    bars = ax.bar(x + hi * width, cos_vals_h, width, label=f'h={h}', alpha=0.8)
    for bar, val in zip(bars, cos_vals_h):
        if val > 0: ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                            f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=90)

ax.set_xticks(x + width/2)
ax.set_xticklabels(sorted_sg_keys, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Goal Cosine Similarity')
ax.set_title(f'B6: SG Predictor Ablation on CEM (K={K_eval}) - {TASK.upper()}')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'b6_cem_ablation_{TASK}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n{"SG Predictor":<14}{"h=5":<14}{"h=10":<14}')
print('-' * 42)
for k in sorted_sg_keys:
    vals = [hier_cem_results.get(f'hier_K{K_eval}_h{h}_{k}_weighted', {}).get('goal_cos', 0) for h in cfg.hier_h_values]
    print(f'{k:<14}{vals[0]:<14.4f}{vals[1]:<14.4f}')


## S16: Save Results

In [ ]:
torch.save({
    'task': TASK,
    'sg_results': {k: {'best_cos': v['best_cos']} for k, v in sg_results.items()},
    'flat_cem_results': flat_cem_results,
    'hier_cem_results': hier_cem_results,
    'best_sg_key': best_sg_key,
    'sg_counts': sg_counts,
    'cfg': {k: v for k, v in cfg.__dict__.items() if not k.startswith('__')},
}, OUTPUT_DIR / f'can_square_hier_{TASK}_results.pt')

print(f'\n{"="*70}')
print(f'{TASK.upper()} EXPERIMENT COMPLETE')
print(f'{"="*70}')
print(f'\nB1: {len(train_sg_local)} episodes, {sum(sg_counts)} subgoals, {np.mean(sg_counts):.2f}/ep')
print(f'B3: Best SG predictor: {best_sg_key} (cos={sg_results[best_sg_key]["best_cos"]:.4f})')
print(f'\nB5 Flat CEM best:')
best_flat = max(flat_cem_results.items(), key=lambda x: x[1].get('goal_cos', 0))
print(f'  {best_flat[0]}: goal_cos={best_flat[1]["goal_cos"]:.4f}, success={best_flat[1]["success_rate"]:.3f}')
print(f'\nB5 Hier CEM best:')
best_hier = max(hier_cem_results.items(), key=lambda x: x[1].get('goal_cos', 0))
print(f'  {best_hier[0]}: goal_cos={best_hier[1]["goal_cos"]:.4f}, success={best_hier[1]["success_rate"]:.3f}')
print(f'\nSaved to outputs/can_square_hier_{TASK}_results.pt')
